# Mini-projet 6 : Atterrissage lunaire

Martin Desombre & Benoît Monnereau-Moinier

### Question 1

$$
v_y^{k+1} = v_y^k + \frac{\Delta t}{2}\left(a_y^k + a_y^{k+1}\right)
$$

$$
y^{k+1} = y^k + v_y^k\,\Delta t
+ \frac{\Delta t^2}{3} a_y^k
+ \frac{\Delta t^2}{6} a_y^{k+1}
$$

_(écriture LateX généré par IA)_

$$
J = \frac{\Delta t}{3}
\sum_{k=0}^{N-1} \Big[
(a_h^k)^2 + (a_h^{k+1})^2 + a_h^k a_h^{k+1}
+ (a_y^k)^2 + (a_y^{k+1})^2 + a_y^k a_y^{k+1}
\Big]
$$

_(écriture LateX généré par IA)_

In [1]:
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from scipy.constants import g as grav

In [20]:
def resolution(T, N, h0, vh0, y0, vy0, amax):
    deltaT = T/N
    nb = N + 1

    ah = ca.SX.sym('ah', nb, 1)
    ay = ca.SX.sym('ay', nb, 1)
    vh = ca.SX.sym('vh', nb, 1)
    vy = ca.SX.sym('vy', nb, 1)
    h = ca.SX.sym('h', nb, 1)
    y = ca.SX.sym('y', nb, 1)

    x = ca.vertcat(ah, ay, vh, vy, h, y)
    print(x.shape)
    lbx = []
    ubx = []

    lbx += [0]*nb
    ubx += [ca.inf]*nb
    lbx += [-ca.inf]*nb
    ubx += [ca.inf]*nb
    lbx += [vh0] + [-ca.inf]*(nb-2) + [0]
    ubx += [vh0] + [ca.inf]*(nb-2) + [0]
    lbx += [vy0] + [-ca.inf]*(nb-2) + [0]
    ubx += [vy0] + [ca.inf]*(nb-2) + [0]
    lbx += [h0] + [-ca.inf]*(nb-2) + [0]
    ubx += [h0] + [ca.inf]*(nb-2) + [0]
    lbx += [y0] + [-ca.inf]*(nb-2) + [0]
    ubx += [y0] + [ca.inf]*(nb-2) + [0]

    assert len(lbx) == len(ubx), "pb longueur lbx vs ubx"
    assert len(lbx) == x.shape[0], "pb longueur x vs bx"
    nbvar = x.shape[0]

    g = []
    lbg = []
    ubg = []

    #dynamique selon h
    for k in range(N):
        g.append(vh[k]-grav*deltaT+(ah[k]+ah[k+1])*deltaT/2-vh[k+1])
        lbg.append(0)
        ubg.append(0)

    deltaTC = deltaT**2
    for k in range(N):
        g.append(h[k]-vh[k]*deltaT-grav*deltaTC/2+deltaTC*ah[k]/3+deltaTC*ah[k+1]/6-h[k+1])
        lbg.append(0)
        ubg.append(0)

    #dynamique selon y
    for k in range(N):
        g.append(vy[k]+(ay[k]+ay[k+1])*deltaT/2 - vy[k+1])
        lbg.append(0)
        ubg.append(0)

    for k in range(N):
        g.append(y[k]+vy[k]*deltaT+ay[k]*deltaTC/3+ay[k+1]*deltaTC/6)
        lbg.append(0)
        ubg.append(0)

    #amax
    for k in range(nb):
        g.append(ah[k]**2 + ay[k]**2)
        lbg.append(0)
        ubg.append(amax**2)

    g = ca.vertcat(*g)

    #cout
    J = (deltaT/3)
    for k in range(N):
        J+=ah[k]**2+ah[k+1]**2+ah[k]*ah[k+1]+ay[k]**2+ay[k+1]**2+ay[k]*ay[k+1]
    
    nlp = {'x': x, 'f' : J, 'g' : g}
    
    solver = ca.nlpsol('s', 'ipopt', nlp)
    sol = solver(
        x0 = np.ones(nbvar),
        lbx = lbx,
        ubx = ubx,
        lbg = lbg,
        ubg = ubg,
    )
    return sol['x'], sol['f']


In [21]:
resolution(60*20, 20, 1000, -41, -1200, 10, 3)

(126, 1)

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:      331
Number of nonzeros in inequality constraint Jacobian.:       42
Number of nonzeros in Lagrangian Hessian.............:       82

Total number of variables............................:      118
                     variables with only lower bounds:       21
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       80
Total number of ine

(DM([5.10586, 10.2075, 10.1948, 10.1821, 10.1694, 10.1567, 10.144, 10.1313, 10.1186, 10.1058, 10.0931, 10.0804, 10.0677, 10.055, 10.0423, 10.0296, 10.0169, 10.0042, 9.99152, 9.97881, 4.98729, 0.409091, 0.181818, -0.0303034, -0.0303034, -0.0303033, -0.0303033, -0.0303033, -0.0303033, -0.0303033, -0.0303032, -0.0303032, -0.0303032, -0.0303031, -0.0303031, -0.030303, -0.0303029, -0.0303028, -0.0303026, -0.0303024, -0.0303019, -0.0151509, -41, -169.999, -146.33, -123.424, -101.28, -79.8978, -59.278, -39.4205, -20.3252, -1.99216, 15.5786, 32.3872, 48.4336, 63.7177, 78.2396, 91.9993, 104.997, 117.232, 128.705, 139.416, 0, 10, 27.7273, 32.2727, 30.4545, 28.6363, 26.8181, 24.9999, 23.1817, 21.3635, 19.5453, 17.7271, 15.9089, 14.0907, 12.2726, 10.4544, 8.6362, 6.81803, 4.99986, 3.18171, 1.36358, 0, 1000, -1940.45, 8973.35, 18444.2, 26517.7, 33239.8, 38656.1, 42812.3, 45754.2, 47527.5, 48178, 47751.3, 46293.3, 43849.6, 40466, 36188.2, 31062, 25133.1, 18447.2, 11050, 0, -1200, -1863.64, -1881.82,